**Задача 5. Более умный источник**
- Модифицировать источник так, чтобы для определенной энергии испускалось случайное число фотонов. Число должно подчиняться распределению Пуассона со средним E*Y.
- Реализовать метод генерации фотонов из задачи 2 как generator (конструкция в python).
- Модифицировать генератор так, чтобы он возвращал (yield) по 100 фотонов вместо одного.

In [1]:
import numpy as np
from numpy import random
from numpy import column_stack
from numpy import sqrt, pi, cos, sin

In [2]:
class Source:
    def __init__(self, Y=10000):
        self.Y = float(Y)

    def generate_photons(self, E):
        expected_N = self.Y * E
        N = random.poisson(expected_N)
        cos_theta = random.uniform(-1.0, 1.0, N)
        phi = random.uniform(0.0, 2 * pi, N)
        sin_theta = sqrt(1.0 - cos_theta ** 2)
        x, y, z = sin_theta * cos(phi), sin_theta * sin(phi), cos_theta
        directions = column_stack((x, y, z))
        return directions

    def generate_photons_stream(self, E):
        expected_N = self.Y * E
        N = random.poisson(expected_N)
        for _ in range(N):
            cos_theta = random.uniform(-1.0, 1.0)
            phi = random.uniform(0.0, 2 * pi)
            sin_theta = sqrt(1.0 - cos_theta ** 2)
            yield np.array([sin_theta * cos(phi), sin_theta * sin(phi), cos_theta])

    def generate_photons_batched(self, E, batch_size=100):
        expected_N = self.Y * E
        N = random.poisson(expected_N)
        n_full_batches, remainder = divmod(N, batch_size)

        def _draw_batch(n):
            cos_theta = random.uniform(-1.0, 1.0, n)
            phi = random.uniform(0.0, 2 * pi, n)
            sin_theta = sqrt(1.0 - cos_theta ** 2)
            x, y, z = sin_theta * cos(phi), sin_theta * sin(phi), cos_theta
            return column_stack((x, y, z))

        for _ in range(n_full_batches):
            yield _draw_batch(batch_size)
        if remainder:
            yield _draw_batch(remainder)

In [3]:
source = Source(Y=10000)
energy = 5.0
directions = source.generate_photons(energy)

print(f"Энергия события: {energy} МэВ")
print(f"Ожидаемое число фотонов: {int(source.Y * energy)}")
print(f"Фактически сгенерировано фотонов: {len(directions)}")

Энергия события: 5.0 МэВ
Ожидаемое число фотонов: 50000
Фактически сгенерировано фотонов: 50220
